# Step 8: LLM-as-Judge Evaluation

This notebook runs the LLM-as-judge (GPT-OSS-120B) evaluation on both macro- and micro-category clusters, scoring sentence membership on a 1-5 Likert scale.

In [ ]:
# © Copyright European Union - 2026

import os
import json
import re
import random
import glob
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"]
)

MODEL_NAME = "gpt-oss-120b"

random.seed(42)


def extract_json_from_reply(raw_output: str):
    """Extract JSON from LLM reply text."""
    try:
        return json.loads(raw_output)
    except (json.JSONDecodeError, TypeError):
        pass
    raw_output = raw_output.strip()
    if "```json" in raw_output:
        raw_output = raw_output.split("```json")[-1].split("```")[0]
    elif "```" in raw_output:
        parts = raw_output.split("```")
        if len(parts) >= 3:
            raw_output = parts[1]
    # Find JSON structure
    for start_char, end_char in [("[", "]"), ("{", "}")]:
        if start_char in raw_output and end_char in raw_output:
            start = raw_output.index(start_char)
            end = raw_output.rindex(end_char) + 1
            try:
                return json.loads(raw_output[start:end])
            except json.JSONDecodeError:
                continue
    return json.loads(raw_output)


def model_json(system_prompt: str, user_message: str) -> dict:
    """Call LLM and return parsed JSON response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return extract_json_from_reply(response.choices[0].message.content)


def model(system_prompt: str, user_message: str) -> str:
    """Call LLM and return raw text response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return response.choices[0].message.content


In [ ]:
DATA_OUT_BASE_PATH = "../out/"
first_level_clusters_fp = f"{DATA_OUT_BASE_PATH}2.merged_clusters.json"
first_level_clusterized_proposals_fp = f"{DATA_OUT_BASE_PATH}3.clusterized_proposals.json"

second_level_clusters_fp = f"{DATA_OUT_BASE_PATH}7.cluster_subcluster.json"
second_level_clusterized_proposals_fp = f"{DATA_OUT_BASE_PATH}7.second_level_clusterized_proposals.json"

first_level_out_fp = f"{DATA_OUT_BASE_PATH}8.llm-judgement-first-level_all.xlsx"
second_level_out_fp = f"{DATA_OUT_BASE_PATH}8.llm-judgement-second-level_all.xlsx"

## First level analysis

In [ ]:
clusterized_proposals = json.load(open(first_level_clusterized_proposals_fp))
clusters = json.load(open(first_level_clusters_fp))

In [ ]:
annotation_lines = []
previous_runs = None
if os.path.exists(first_level_out_fp):
    previous_runs = pd.read_excel(first_level_out_fp)
for cluster_name, cluster_proposals in tqdm(clusterized_proposals.items(), desc="clusters"):
    cluster_summary = [x['summary'] for x in clusters if x['name'] == cluster_name][0]


    for proposal in tqdm(cluster_proposals):
        if previous_runs is None or len(previous_runs[previous_runs['id'] == proposal['id']]) == 0:
            proposal['cluster'] = cluster_name
            proposal['summary'] = cluster_summary
            prompt_cluster = f"""Your task is to decide wether a sentence belong to a group, given the group title and its description.
                Give a score from 1 to 5 to judge if the sentence belongs to the group, where 1 means that the sentence is not belonging to the set, while 5 means that it definitely belongs to it.

                Provide in output a JSON with the following structure, do not output anything else.

                class Score(BaseModel):
                    score: int # The score indicating if the sentence belongs to the group identified by title and description. The score can range in the continuum from 1 to 5. Here 1 means that the sentence is not belonging to the set, while 5 means that it definitely belongs to it.

                Output:
                    Score # the score assigned to the sentence
            """
            text_prompt = f"""
                Here is the group title:
                {cluster_name}
                Here is the group description:
                {cluster_summary}

                Here is the sentence:
                {proposal['title']}
            """
            r = model_json(prompt_cluster, text_prompt)
            if r and r.get('score'):
                proposal["Q3"] = r['score']
                annotation_lines.append(proposal)
        else:
            annotation_lines.append(previous_runs[previous_runs['id'] == proposal['id']].iloc[0].to_dict())
    
    pd.DataFrame(annotation_lines).to_excel(first_level_out_fp, index=False)
pd.DataFrame(annotation_lines).to_excel(first_level_out_fp, index=False)

## Second level analysis

In [ ]:
second_level_clusterized_proposals  = {k:v for x in json.load(open(second_level_clusterized_proposals_fp)).values() for k,v in x.items()}
second_level_clusters = [x for subclusters in json.load(open(second_level_clusters_fp)).values() for x in subclusters]

In [ ]:
annotation_lines = []
previous_runs = None
if os.path.exists(second_level_out_fp):
    previous_runs = pd.read_excel(second_level_out_fp)
for cluster_name, cluster_proposals in tqdm(second_level_clusterized_proposals.items(), desc="clusters"):
    cluster_summary = [x['summary'] for x in second_level_clusters if x['name'] == cluster_name][0]


    for proposal in tqdm(cluster_proposals):
        if previous_runs is None or len(previous_runs[previous_runs['id'] == proposal['id']]) == 0:
            proposal['cluster'] = cluster_name
            proposal['summary'] = cluster_summary
            prompt_cluster = f"""Your task is to decide wether a sentence belong to a group, given the group title and its description.
                Give a score from 1 to 5 to judge if the sentence belongs to the group, where 1 means that the sentence is not belonging to the set, while 5 means that it definitely belongs to it.

                Provide in output a JSON with the following structure, do not output anything else.

                class Score(BaseModel):
                    score: int # The score indicating if the sentence belongs to the group identified by title and description. The score can range in the continuum from 1 to 5. Here 1 means that the sentence is not belonging to the set, while 5 means that it definitely belongs to it.

                Output:
                    Score # the score assigned to the sentence
            """
            text_prompt = f"""
                Here is the group title:
                {cluster_name}
                Here is the group description:
                {cluster_summary}

                Here is the sentence:
                {proposal['title']}
            """
            r = model_json(prompt_cluster, text_prompt)
            if r and r.get('score'):
                proposal["Q3"] = r['score']
                annotation_lines.append(proposal)
        else:
            annotation_lines.append(previous_runs[previous_runs['id'] == proposal['id']].iloc[0].to_dict())
    
    pd.DataFrame(annotation_lines).to_excel(second_level_out_fp, index=False)
pd.DataFrame(annotation_lines).to_excel(second_level_out_fp, index=False)

## Average scores

In [ ]:
level = "second"
# load data
if level == "first":
    df = pd.read_excel(first_level_out_fp)
else:
    df = pd.read_excel(second_level_out_fp)

score_column = 'Q3'
scores_cluster = {}
scores_cluster_lang = {}
# Overall
scores = []
lang_scores = {}

for idx, row in df.iterrows():
    cluster_title = row['cluster']
    language = row['language']

    if cluster_title not in scores_cluster:
        scores_cluster[cluster_title] = []
    # Lang
    if cluster_title not in scores_cluster_lang:
        scores_cluster_lang[cluster_title] = {}
    if language not in scores_cluster_lang[cluster_title]:
        scores_cluster_lang[cluster_title][language] = []
    if language not in lang_scores:
        lang_scores[language] = []

    scores_cluster[cluster_title].append(row[score_column])
    scores_cluster_lang[cluster_title][language].append(row[score_column])
    scores.append(row[score_column])
    lang_scores[language].append(row[score_column])
# End for

results = []
for cluster_title in scores_cluster:
    line = {
        'cluster_name': cluster_title,
        'AVG': np.mean(scores_cluster[cluster_title]),
        'Support': len(scores_cluster[cluster_title]),
    }
    for lang in scores_cluster_lang[cluster_title]:
        line[lang.upper()] = np.mean(scores_cluster_lang[cluster_title][lang])
    results.append(line)

# Overall metrics
line = {
    'cluster_name': "Overall",
    'AVG': np.mean(scores),
    'Support': len(scores),
}
for lang in lang_scores:
    line[lang.upper()] = np.mean(lang_scores[lang])
results.append(line)

result_df = pd.DataFrame(results)
result_df = result_df.round(2)
print(list(lang_scores.keys()))
